In [1]:
import nltk
from nltk import word_tokenize

In [2]:
text="if you say anything wrong"
tokens=word_tokenize(text)
tokens

['if', 'you', 'say', 'anything', 'wrong']

In [ ]:
# predict.py

import joblib
import pandas as pd
import numpy as np
from datetime import datetime


model = joblib.load("aqi_model.pkl")


def get_user_input():
    print("\nEnter current values:\n")

    data = {}

    # Current values
    data["pm25"] = float(input("PM2.5: "))
    data["pm10"] = float(input("PM10: "))
    data["no2"] = float(input("NO2: "))
    data["so2"] = float(input("SO2: "))
    data["co"] = float(input("CO: "))
    data["o3"] = float(input("O3: "))
    data["temperature"] = float(input("Temperature: "))
    data["humidity"] = float(input("Humidity: "))
    data["wind_speed"] = float(input("Wind Speed: "))
    data["visibility"] = float(input("Visibility: "))

    # Past AQI (MANDATORY)
    print("\n--- Past AQI values (important) ---")
    data["AQI_lag_1"] = float(input("AQI (t-1 hour): "))
    data["AQI_lag_24"] = float(input("AQI (t-24 hours): "))

    return data


def build_features(data):
    now = datetime.now()

    df = pd.DataFrame([data])

    # Time features
    df["hour"] = now.hour
    df["day"] = now.day
    df["month"] = now.month
    df["weekday"] = now.weekday()

    # Cyclical
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    df["dow_sin"] = np.sin(2 * np.pi * df["weekday"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["weekday"] / 7)

    # Interaction
    df["pm25_no2"] = df["pm25"] * df["no2"]
    df["pm25_o3"] = df["pm25"] * df["o3"]

    # Approx rolling (hack — not perfect but usable)
    df["AQI_roll_3"] = df["AQI_lag_1"]
    df["AQI_roll_6"] = df["AQI_lag_1"]
    df["AQI_roll_12"] = df["AQI_lag_24"]

    df["AQI_std_3"] = 0
    df["AQI_std_6"] = 0

    # Missing lags → approximate
    for lag in [2, 3, 6, 12, 48, 72, 168]:
        df[f"AQI_lag_{lag}"] = df["AQI_lag_1"]

    pollutants = ["pm25", "pm10", "no2", "so2", "co", "o3"]
    for col in pollutants:
        for lag in [1, 3, 6]:
            df[f"{col}_lag_{lag}"] = df[col]

    # Drop unused columns alignment fix will be next

    return df


def align_features(df):
    # Load training columns
    import joblib
    cols = joblib.load("feature_columns.pkl")

    for col in cols:
        if col not in df.columns:
            df[col] = 0

    df = df[cols]
    return df


def main():
    data = get_user_input()
    df = build_features(data)

    df = align_features(df)

    pred = model.predict(df)

    print("\n🔥 Predicted AQI (next hour):", round(pred[0], 2))


if __name__ == "__main__":
    main()